# T1. The model is data

What is a SysML v2 model once `longeron` parses it?

The answer is data. `longeron` parses SysML v2 text into a tree of plain
Python dataclasses. The rest of this notebook shows what follows from that
fact:

- You can walk the tree and read typed fields.
- You can build new elements from the same dataclasses.
- The model survives a JSON round trip without loss.
- One `save` call writes SysML text, JSON, or KerML.
- `validate` checks the tree before you trust it.

The subject is the drone from `examples/drone.sysml`. Every tutorial in this
series builds on the same drone. The first cell loads the drone and prints
one line per member of its package.

In [ ]:
import longeron

drone = longeron.load("../examples/drone.sysml")

for element in drone.find("Drone").members:
    if not element.name:
        continue
    nested = sum(1 for _ in element.iter_tree()) - 1
    print(f"{element.kind:12s} {element.name:18s} {nested:3d} nested elements")
print("total:", sum(1 for _ in drone.iter_tree()), "elements")

## Every element is a typed dataclass

One file holds the whole drone program. The listing above spans parts,
calcs, requirements, an action, and two state machines. Each printed line
is an element. Each element is a dataclass with a `kind`, a
`qualified_name`, and typed fields. Closed vocabularies such as `kind` are
`typing.Literal` aliases, so static checkers can read them.

`find` looks up one element by qualified name. The next cell reads
`maxCruiseSpeed`, the drone's computed max cruise speed. Its value is an
expression tree, not a number. Tutorial T2 evaluates it.

The same dataclasses are the authoring API. The second cell below builds a
long-range battery without any SysML text. It adds the new part definition
to the package. `to_sysml` prints any element back as text, so both
authoring routes meet in one object model.

In [ ]:
speed = drone.find("Drone::QuadCopter::maxCruiseSpeed")
print("kind:      ", speed.kind)
print("types:     ", speed.types)
print("owner:     ", speed.owner.qualified_name)
print("value expr:", speed.value.expr.to_text())
print()
print("Motor doc: ", drone.find("Drone::Motor").doc)

In [ ]:
from longeron import model as M

battery = M.Definition(kind="part", name="LongRangeBattery", supers=["Battery"])
battery.add(
    M.Usage(
        kind="attribute",
        name="capacity",
        types=["Real"],
        value=M.FeatureValue(longeron.parse_expression("8000.0")),
    ),
    M.Usage(
        kind="attribute",
        name="mass",
        types=["Real"],
        value=M.FeatureValue(longeron.parse_expression("0.55")),
    ),
)
drone.find("Drone").add(battery)

assert drone.find("Drone::LongRangeBattery") is battery
print(longeron.to_sysml(battery))

## The JSON round trip is lossless

The claim is checkable, so the next cell checks it. `to_json` writes the
tree. `from_json` reads it back. The clone must equal the original,
dictionary for dictionary. The clone must also still execute. The cell
calls the drone's `HoverTime` calc on the clone, with the capacity of the
battery you just built.

In [ ]:
clone = longeron.from_json(longeron.to_json(drone))

assert longeron.to_dict(clone) == longeron.to_dict(drone)
print("round trip preserved all", sum(1 for _ in clone.iter_tree()), "elements")

minutes = longeron.Interpreter(clone).call("Drone::HoverTime", capacity=8000.0)
print("HoverTime on the clone:", minutes, "minutes")

## One `save` call, three formats

`save` dispatches on the file suffix:

- `.sysml` writes the textual notation.
- `.json` writes the lossless schema from the previous cell.
- `.kerml` writes a projection onto KerML, the kernel language beneath
  SysML v2.

The KerML projection is one-way. The bundled KerML grammar re-parses the
output, and the next cell proves that with `parse_kerml_text`.

`load` accepts one file or a directory. A directory load merges every file
under one root namespace, so imports resolve across files. The second cell
below splits a battery excerpt of the drone across two files. A calc in one
file then reads attributes of a part defined in the other.

In [ ]:
import tempfile
from pathlib import Path

out = Path(tempfile.mkdtemp())
for suffix in (".sysml", ".json", ".kerml"):
    longeron.save(drone, out / f"drone{suffix}")
print(sorted(path.name for path in out.iterdir()))

kerml_text = (out / "drone.kerml").read_text()
longeron.parse_kerml_text(kerml_text)  # raises on invalid KerML
print()
print("\n".join(kerml_text.splitlines()[:6]))

In [ ]:
workspace = Path(tempfile.mkdtemp())
(workspace / "batteries.sysml").write_text(
    "package Batteries { part def LiPo3S {"
    " attribute capacity : Real = 5200.0; attribute voltage : Real = 11.1; } }"
)
(workspace / "power.sysml").write_text("""
package Power {
    private import Batteries::*;
    part def Pack { part cell : LiPo3S; }
    calc def PackEnergy {
        in pack : Pack;
        return : Real = pack.cell.capacity / 1000.0 * pack.cell.voltage;
    }
}
""")

merged = longeron.load(workspace)  # directory -> one merged model
interp = longeron.Interpreter(merged)
pack = interp.instantiate("Power::Pack")
print("pack energy:", interp.call("Power::PackEnergy", pack=pack), "Wh")

## The first quality gate

`validate` returns one diagnostic per problem. Structural problems are
errors. Unresolved references are warnings. The validator knows the
standard library, so a bare `Real` resolves without an import. The command
line exposes the same gate as `longeron lint <file>`.

The drone model validates clean, and the first assert checks that. The
second model plants three defects in a package of field modifications.
Each defect prints as one diagnostic.

In [ ]:
assert not longeron.validate(drone), "the drone model must validate clean"
print("drone: 0 diagnostics")

buggy = longeron.loads("""
package FieldMods {
    part def LandingSkid;
    part def LandingSkid;                    // duplicate name
    part spare : NoSuchBattery;              // dangling reference
    part def CameraMount {
        attribute mass : Real = 0.02;
        attribute margin : Real = maas + 0.01;   // typo in the expression
    }
}
""")
for diagnostic in longeron.validate(buggy):
    print(diagnostic)

## The answer

Once `longeron` parses it, a SysML v2 model is a tree of typed dataclasses.
In this notebook you:

- walked the drone's package and read typed fields
- built a new part definition from the same dataclasses
- proved the JSON round trip lossless
- saved the model as SysML text, JSON, and KerML
- validated the model and read three planted diagnostics

So far the tree only holds data. The drone's motor bench table also
backs a claimed max cruise speed of 20.0 m/s. Tutorial T2, "The model executes", makes the
model compute that number.